In [ ]:
#producent
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime
 
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
 
sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']
 
def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }
 
for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['user_id']} | {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)
 
producer.flush()
producer.close()

In [ ]:
#konsument
%%file consumer.py
from kafka import KafkaConsumer
import json
from datetime import datetime, timedelta
from collections import defaultdict

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję anomalii prędkości ( >3 transakcje / 60s )...")
user_events = defaultdict(list)
window = 60
limit = 3

for message in consumer:
    tx = message.value
    user_id = tx["user_id"]
    tx_time = datetime.fromisoformat(tx["timestamp"])
    user_events[user_id].append(tx_time)
    cutoff = tx_time - timedelta(seconds=window)
    user_events[user_id] = [
        t for t in user_events[user_id] if t >= cutoff
    ]

    if len(user_events[user_id]) > limit:
        print(
            f"ALERT: user_id={user_id} | "
            f"liczba transakcji={len(user_events[user_id])} | "
            f"okno={window}s"
        )